# SQL query from table names - Continued

In [10]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [9]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [11]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [30]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [15]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [35]:
#new
context_user = context.copy()
print(return_CCRMSQL("""YOUR QUERY HERE""", context_user))

```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
ORDER BY s.salary DESC
LIMIT 1;
```


In [37]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee who is best paid by joining the "employees" table with the "salary" table on the ID_usr field. It then orders the result by salary in descending order and limits the output to only one row, which corresponds to the employee with the highest salary.


In [13]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context_user))

NameError: name 'context_user' is not defined

In [39]:
#old
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql
SELECT s.Institution
FROM studies s
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY AVG(sa.salary) DESC
LIMIT 1;
```

This SQL query joins the "studies" and "salary" tables on the ID_usr column. It then calculates the average salary for each institution, orders the results in descending order based on the average salary, and returns the institution with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [16]:
base_context_v1 = [
    {
        "role": "system",
        "content": """
You are a SQL table selection assistant.

Available tables:

employees: employee personal information (id, name, department)
salary: employee salary per year (employee_id, year, amount)
studies: education background (employee_id, degree, field_of_study)

Select the necessary tables to answer the question.
Return only table names separated by commas.
"""
    }
]

In [17]:
base_context_v2 = [
    {
        "role": "system",
        "content": """
You are a SQL table selection system.

Rules:
- Only use the tables provided.
- Do NOT invent tables.
- Select the MINIMUM necessary tables.
- Return strictly valid JSON:
{"tables": ["table1", "table2"]}

Available tables:

employees: employee personal information (id, name, department)
salary: employee salary per year (employee_id, year, amount)
studies: education background (employee_id, degree, field_of_study)
"""
    }
]

In [18]:
base_context_v3 = [
    {
        "role": "system",
        "content": """
You are a table selection engine.

Example:

Tables:
employees: employee information
salary: salary information
studies: education information

Question:
Who earns the highest salary?

Answer:
["employees", "salary"]

Now use the same logic for the next question.

Available tables:

employees: employee personal information (id, name, department)
salary: employee salary per year (employee_id, year, amount)
studies: education background (employee_id, degree, field_of_study)

Return only the list of tables.
"""
    }
]

In [19]:
questions = [
    "Which employee earns the highest salary?",
    "List employees and their degree.",
    "What is the average salary in 2022?",
    "Which employees studied engineering and earn more than 50000?"
]

In [22]:
def test_all_versions():
    for q in questions:
        print("====================================")
        print("QUESTION:", q)
        print("====================================\n")
        
        print("---- Version 1 ----")
        print(return_CCRMSQL(q, base_context_v1))
        print()
        
        print("---- Version 2 ----")
        print(return_CCRMSQL(q, base_context_v2))
        print()
        
        print("---- Version 3 ----")
        print(return_CCRMSQL(q, base_context_v3))
        print("\n\n")

In [24]:
test_all_versions()

QUESTION: Which employee earns the highest salary?

---- Version 1 ----
employees, salary

---- Version 2 ----
{"tables": ["salary"]}

---- Version 3 ----
["employees", "salary"]



QUESTION: List employees and their degree.

---- Version 1 ----
employees, studies

---- Version 2 ----
{"tables": ["employees", "studies"]}

---- Version 3 ----
["employees", "studies"]



QUESTION: What is the average salary in 2022?

---- Version 1 ----
salary

---- Version 2 ----
{"tables": ["salary"]}

---- Version 3 ----
["salary"]



QUESTION: Which employees studied engineering and earn more than 50000?

---- Version 1 ----
employees, salary, studies

---- Version 2 ----
{"tables": ["employees", "salary", "studies"]}

---- Version 3 ----
["employees", "salary", "studies"]





The basic system prompt (V1) generally selected correct tables but sometimes returned additional explanatory text.

Without strict formatting constraints, output consistency varied across questions.

The model occasionally over-selected tables when joins were implied but not strictly necessary.

Adding explicit JSON constraints (V2) significantly improved structural consistency.

The “minimum necessary tables” rule reduced over-selection errors.

The few-shot prompt (V3) produced the most stable and semantically correct selections.

Clear constraints in the system message were more impactful than longer table descriptions.